# Naive RAG — "Os Sertões", de Euclides da Cunha

Este notebook implementa a abordagem mais simples de RAG (*Retrieval-Augmented Generation*), usada como referência para comparação com o **Parent RAG** e o **Rerank RAG**.

**Fluxo:**
1. Carregar o PDF do livro e dividi-lo em pedaços (*chunks*).
2. Gerar embeddings dos chunks e armazená-los em um banco vetorial (Chroma).
3. Para cada pergunta, recuperar os 3 chunks mais similares.
4. Enviar esses chunks como contexto para o LLM gerar a resposta.

É "naive" porque os chunks recuperados vão direto para o LLM, sem reordenação (*rerank*) e sem recuperar documentos maiores (*parent documents*).

## Usando arquivo .env para controlar variaveis de ambiente
Para evitar exposição da chave `OPENAI_API_KEY` optei por utilizar arquivo `.env` com a informação da chave.

Para seguir o mesmo método basta criar um arquivo `.env` no mesmo diretório do arquivo `naive_rag.ipynb`.
A importação da chave será feita através da célula abaixo que faz a instalação de um biblioteca para carregar
os valores do arquivo `.env`.

In [11]:
%pip install python-dotenv
import dotenv
%load_ext dotenv
%dotenv

Note: you may need to restart the kernel to use updated packages.
The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Importações
Bibliotecas do LangChain para carregar o PDF, dividir o texto, gerar embeddings, criar o banco vetorial e montar a cadeia de prompt → LLM → saída.

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser

## Perguntas do desafio
As cinco perguntas sobre "Os Sertões" definidas no enunciado. As mesmas perguntas são usadas nos três notebooks para permitir a comparação.

In [ ]:
# Perguntas
questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em \"Os Sertões\"? Como esses aspectos refletem a visão do autor sobre o Brasil da época?",
]

## Modelos
- **Embeddings:** `OpenAIEmbeddings` transforma texto em vetores para a busca por similaridade.
- **LLM:** `gpt-3.5-turbo` gera as respostas. `max_tokens=200` limita o tamanho da resposta e pode cortá-la antes do fim.

In [ ]:
# Carregando o Modelo de LLM 
embeddings_model = OpenAIEmbeddings() 
llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    max_tokens=200,              
)

## Carregamento do PDF
O `PyPDFLoader` lê o arquivo `os-sertoes.pdf` e retorna um documento por página.

In [ ]:
# Carregando o PDF
pdf_path = "os-sertoes.pdf" 

loader = PyPDFLoader(pdf_path, extract_images=False)

pages = loader.load_and_split()


## Chunking e banco vetorial
As páginas são divididas em chunks de até **4000 caracteres** com **20 caracteres de sobreposição**. Chunks grandes levam mais contexto por trecho, mas tornam a busca menos precisa. A sobreposição pequena pode cortar ideias na fronteira entre chunks.

Os chunks são convertidos em embeddings e armazenados em memória no **Chroma**.

In [ ]:
# Separando o documento em pedaços
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,          
    chunk_overlap=20,        
    length_function=len,      
    add_start_index=True,   
)

chunks = text_splitter.split_documents(pages)

db = Chroma.from_documents(chunks, embedding=embeddings_model)

## Recuperador (*retriever*)
Busca por similaridade que retorna os **3 chunks** mais próximos da pergunta.

In [ ]:
# Configurando o recuperador
retriever = db.as_retriever(search_kwargs={"k": 3})


## Prompt e cadeia de execução
O prompt define o papel do modelo (especialista em literatura brasileira) e recebe o `context` recuperado e a `question`.

A cadeia `prompt | llm | parser` formata o prompt, chama o LLM e converte a saída em texto.

In [18]:
TEMPLATE = """
Você é um especialista em literatura brasileira. Responda a pergunta abaixo utilizando o contexto informado

Contexto: {context}
    
Pergunta: {question}
"""

prompt = PromptTemplate(input_variables=["context", "question"], template=TEMPLATE)
parser = StrOutputParser()
sequence = RunnableSequence(prompt | llm | parser)

## Função de resposta
Para cada pergunta: recupera o contexto com o retriever e executa a cadeia com contexto + pergunta.

In [ ]:
# Função para responder a uma pergunta no contexto 
def answer_question(question: str):

    context = retriever.invoke(question)

    response = sequence.invoke({"context": context, "question": question})

    return response

## Execução
Responde às cinco perguntas e exibe número, pergunta e resposta.

In [ ]:
# Iterando sobre as perguntas
for index, question in enumerate(questions):
  
    resposta = answer_question(question)
 
    print({"numero": index, "pergunta": question, "resposta": resposta})

{'numero': 0, 'pergunta': 'Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?', 'resposta': 'Euclides da Cunha descreve o ambiente natural do sertão nordestino como um lugar de extrema dificuldade e desafio, onde a vida é marcada pela aridez, pela falta de recursos e pela luta constante pela sobrevivência. Ele destaca a importância da terra fértil na região, que libertou os indígenas, mas também menciona a cultura extensiva da cana, importada da Madeira, que determinou o esquecimento dos sertões.\n\nEssa realidade do sertão nordestino influencia diretamente a vida dos habitantes, que são obrigados a enfrentar condições adversas, como a escassez de água, a falta de infraestrutura e a ausência de oportunidades. Euclides da Cunha mostra como essas dificuldades moldam o caráter e a cultura das pessoas que viv'}
{'numero': 1, 'pergunta': 'Quais são as principais características da população sertaneja descritas por E

## Observações e limitações
- **Sem reordenação:** a similaridade de embeddings nem sempre traz os trechos mais relevantes nas primeiras posições. O Rerank RAG tenta resolver isso.
- **Chunk único para busca e contexto:** o mesmo chunk serve para buscar e para responder. O Parent RAG busca com chunks pequenos (mais precisos) e envia ao LLM o documento pai (mais contexto).
- **Respostas curtas:** `max_tokens=200` pode truncar respostas para perguntas abertas.
- **Contexto com metadados:** os `Document` recuperados são passados direto ao prompt, então metadados (página, fonte) também entram no contexto.